# B1_S00 — Can You Beat Randomness?

## Opening hook

**Can you beat randomness?**

EuroMillions looks simple: 5 main numbers, 2 Lucky Stars, one draw, one dream.

But before learning pandas, visualization, or machine learning, students need a more important lesson:

> Humans are pattern-making machines. Randomness is where that instinct becomes dangerous.

Questions for the room:

- Do some numbers come out more often?
- Are there hot numbers?
- Are there cursed numbers?
- Can we detect a real pattern?
- Or are we just seeing shapes in noise?

This notebook is **not** about predicting lottery numbers. It is about learning why Data Science often starts by proving that a pattern is fake.

## Decision problem

When we see historical data, how do we know whether a visible pattern is meaningful or just randomness doing what randomness does?

Dataset: **EuroMillions Historical Data** by Duarte Pereira da Cruz on Kaggle. It contains historical EuroMillions draws from 2004 to present, including draw dates, five main numbers, two Lucky Stars, and jackpot/rollover context when available.

In [ ]:
from pathlib import Path
import re
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "outputs").exists() or (candidate / "notebooks").exists():
        ROOT = candidate
        break

OUT = ROOT / "outputs" / "bloc1" / "can_you_beat_randomness"
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
print("Output folder:", OUT)

## Load the Kaggle dataset

On Kaggle, add the dataset **EuroMillions Historical Data** to the notebook input. Locally, place the CSV anywhere under `data/`, `datasets/`, `kaggle/`, or the repo root.

The notebook intentionally fails with a clear message if the dataset is missing. We do not generate fake lottery history.

In [ ]:
def find_euromillions_csv() -> Path:
    roots = [Path("/kaggle/input"), ROOT / "data", ROOT / "datasets", ROOT / "kaggle", ROOT]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob("*.csv"))
    scored = []
    for p in candidates:
        name = p.name.lower()
        full = str(p).lower()
        score = 0
        if "euromillion" in name or "euromillions" in name:
            score += 5
        if "euromillion" in full or "euromillions" in full:
            score += 3
        if "duarte" in full or "cruz" in full:
            score += 2
        if score:
            scored.append((score, p))
    if not scored:
        raise FileNotFoundError(
            "EuroMillions CSV not found. On Kaggle, attach dataset: "
            "duartepereiradacruz/euromillions-historical-data. "
            "Locally, place euromillions.csv under data/ or datasets/."
        )
    return sorted(scored, reverse=True)[0][1]

csv_path = find_euromillions_csv()
raw = pd.read_csv(csv_path)
print(csv_path)
raw.head()

## Prepare draw columns

Kaggle datasets can evolve. Instead of hard-coding one schema too aggressively, we detect likely main-number and Lucky Star columns from names and values.

In [ ]:
def normalize_column(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")

df = raw.copy()
df.columns = [normalize_column(c) for c in df.columns]

# Date column: prefer columns containing date/draw.
date_candidates = [c for c in df.columns if "date" in c or "draw" in c]
date_col = date_candidates[0] if date_candidates else df.columns[0]
df[date_col] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
if df[date_col].notna().sum() == 0:
    raise ValueError(f"Could not parse draw dates from column: {date_col}")

numeric_cols = []
for c in df.columns:
    if c == date_col:
        continue
    converted = pd.to_numeric(df[c], errors="coerce")
    if converted.notna().mean() > 0.8:
        df[c] = converted
        numeric_cols.append(c)

# Prefer semantic column names, then fallback to value ranges.
star_name_cols = [c for c in numeric_cols if any(token in c for token in ["star", "lucky", "etoile", "estrela"])]
main_name_cols = [c for c in numeric_cols if any(token in c for token in ["ball", "number", "num", "n_"]) and c not in star_name_cols]

if len(star_name_cols) >= 2:
    star_cols = star_name_cols[:2]
else:
    possible_stars = [c for c in numeric_cols if df[c].between(1, 12).mean() > 0.95]
    star_cols = possible_stars[-2:]

if len(main_name_cols) >= 5:
    main_cols = [c for c in main_name_cols if c not in star_cols][:5]
else:
    possible_main = [c for c in numeric_cols if c not in star_cols and df[c].between(1, 50).mean() > 0.95]
    main_cols = possible_main[:5]

if len(main_cols) != 5 or len(star_cols) != 2:
    raise ValueError(f"Could not detect 5 main number columns and 2 Lucky Star columns. Columns: {list(df.columns)}")

draws = df[[date_col] + main_cols + star_cols].dropna().copy()
draws = draws.rename(columns={date_col: "draw_date"})
draws[main_cols + star_cols] = draws[main_cols + star_cols].astype(int)
draws = draws.sort_values("draw_date").reset_index(drop=True)

print("Main columns:", main_cols)
print("Lucky Star columns:", star_cols)
print("Draws:", len(draws), "from", draws["draw_date"].min().date(), "to", draws["draw_date"].max().date())
draws.head()

# 1. Frequency analysis

The first temptation is simple: count which numbers came out most often.

This is where intuition starts whispering: “that number wins more.”

The statistical question is sharper:

> Is the difference surprising, or is it normal variation?

In [ ]:
main_values = draws[main_cols].to_numpy().ravel()
star_values = draws[star_cols].to_numpy().ravel()

main_freq = pd.Series(main_values).value_counts().reindex(range(1, 51), fill_value=0).rename_axis("number").reset_index(name="frequency")
star_freq = pd.Series(star_values).value_counts().reindex(range(1, 13), fill_value=0).rename_axis("lucky_star").reset_index(name="frequency")

main_freq["share"] = main_freq["frequency"] / main_freq["frequency"].sum()
star_freq["share"] = star_freq["frequency"] / star_freq["frequency"].sum()

main_freq.to_csv(OUT / "main_number_frequencies.csv", index=False)
star_freq.to_csv(OUT / "lucky_star_frequencies.csv", index=False)

main_freq.sort_values("frequency", ascending=False).head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(main_freq["number"], main_freq["frequency"])
axes[0].axhline(main_freq["frequency"].mean(), color="red", linestyle="--", label="expected average")
axes[0].set_title("Main number frequencies")
axes[0].set_xlabel("Number")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].bar(star_freq["lucky_star"], star_freq["frequency"], color="goldenrod")
axes[1].axhline(star_freq["frequency"].mean(), color="red", linestyle="--", label="expected average")
axes[1].set_title("Lucky Star frequencies")
axes[1].set_xlabel("Lucky Star")
axes[1].legend()

fig.tight_layout()
fig.savefig(OUT / "frequency_analysis.png", dpi=140)
plt.show()

## Does one number really “win more”?

A number can be above average without being special.

In random systems, someone is always first. Someone is always last. That does not mean the leader has a hidden force.

In [ ]:
frequency_summary = pd.DataFrame({
    "metric": ["main_mean", "main_std", "main_min", "main_max", "star_mean", "star_std", "star_min", "star_max"],
    "value": [
        main_freq["frequency"].mean(), main_freq["frequency"].std(), main_freq["frequency"].min(), main_freq["frequency"].max(),
        star_freq["frequency"].mean(), star_freq["frequency"].std(), star_freq["frequency"].min(), star_freq["frequency"].max(),
    ]
}).round(3)
frequency_summary.to_csv(OUT / "frequency_summary.csv", index=False)
frequency_summary

# 2. Hot numbers vs cold numbers

Hot/cold lists are seductive because they turn noise into a story.

But a story is not evidence.

In [ ]:
def overdue_table(values_frame, cols, max_value, label):
    rows = []
    for n in range(1, max_value + 1):
        mask = values_frame[cols].eq(n).any(axis=1)
        last_seen = values_frame.loc[mask, "draw_date"].max()
        draws_since = int((values_frame["draw_date"] > last_seen).sum()) if pd.notna(last_seen) else len(values_frame)
        rows.append({label: n, "last_seen": last_seen, "draws_since_seen": draws_since})
    return pd.DataFrame(rows)

main_overdue = overdue_table(draws, main_cols, 50, "number")
star_overdue = overdue_table(draws, star_cols, 12, "lucky_star")

hot_cold = {
    "main_hot": main_freq.sort_values("frequency", ascending=False).head(10).to_dict("records"),
    "main_cold": main_freq.sort_values("frequency", ascending=True).head(10).to_dict("records"),
    "main_overdue": main_overdue.sort_values("draws_since_seen", ascending=False).head(10).to_dict("records"),
    "star_hot": star_freq.sort_values("frequency", ascending=False).head(5).to_dict("records"),
    "star_cold": star_freq.sort_values("frequency", ascending=True).head(5).to_dict("records"),
}

(OUT / "hot_cold_numbers_summary.json").write_text(json.dumps(hot_cold, indent=2, default=str))
main_freq.sort_values("frequency", ascending=False).head(5), main_freq.sort_values("frequency").head(5), main_overdue.sort_values("draws_since_seen", ascending=False).head(5)

## Gambler’s fallacy

The gambler’s fallacy says: “This number has not appeared for a while, so it is due.”

But in a fair draw, the machine does not remember.

An overdue number is not a promise. It is a feeling.

# 3. Human bias

Even if randomness is fair, humans can still play badly.

Why?

- **Birthdays effect:** many people choose 1–31, ignoring 32–50.
- **Favorite numbers:** people cluster around memorable numbers.
- **Pattern preference:** sequences, diagonals, symmetry, and “nice looking” tickets feel meaningful.
- **Narrative bias:** we invent explanations after seeing results.

This does not change the draw probability. It changes how many people may share a prize if common numbers win.

In [ ]:
birthday_share_observed = (pd.Series(main_values).between(1, 31).mean())
birthday_share_uniform = 31 / 50
human_bias = pd.DataFrame([
    {"choice_zone": "1-31 birthday range", "observed_share": birthday_share_observed, "uniform_expectation": birthday_share_uniform},
    {"choice_zone": "32-50 non-birthday range", "observed_share": 1-birthday_share_observed, "uniform_expectation": 1-birthday_share_uniform},
]).round(3)
human_bias.to_csv(OUT / "birthday_effect_note.csv", index=False)
human_bias

Reflection question:

Even if every combination has the same probability, why might a human-selected ticket be strategically worse than a random ticket?

# 4. Monte Carlo simulation

Now we compare intuition with pure randomness.

If the real draw is fair, simulated random draws should also produce “hot” numbers, “cold” numbers, and overdue numbers.

That is the point: randomness naturally creates patterns that look suspicious.

In [ ]:
N_SIM = len(draws)
N_RUNS = 1_000
rng = np.random.default_rng(2026)

sim_max_minus_min = []
sim_top_frequency = []
for _ in range(N_RUNS):
    simulated = np.vstack([rng.choice(np.arange(1, 51), size=5, replace=False) for _ in range(N_SIM)])
    freq = pd.Series(simulated.ravel()).value_counts().reindex(range(1, 51), fill_value=0)
    sim_max_minus_min.append(int(freq.max() - freq.min()))
    sim_top_frequency.append(int(freq.max()))

observed_spread = int(main_freq["frequency"].max() - main_freq["frequency"].min())
observed_top = int(main_freq["frequency"].max())

simulation_report = pd.DataFrame({
    "metric": ["observed_max_minus_min", "simulation_mean_max_minus_min", "simulation_p95_max_minus_min", "observed_top_frequency", "simulation_p95_top_frequency"],
    "value": [observed_spread, np.mean(sim_max_minus_min), np.percentile(sim_max_minus_min, 95), observed_top, np.percentile(sim_top_frequency, 95)],
}).round(3)
simulation_report.to_csv(OUT / "simulation_report.csv", index=False)
simulation_report

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sim_max_minus_min, bins=30, alpha=0.8, label="random simulations")
ax.axvline(observed_spread, color="red", linewidth=2, label="observed EuroMillions history")
ax.set_title("Random simulations also create hot/cold spreads")
ax.set_xlabel("Max frequency - min frequency")
ax.set_ylabel("Simulation count")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "monte_carlo_randomness_spread.png", dpi=140)
plt.show()

# 5. Statistical interpretation

Key ideas:

**Law of large numbers**  
As the number of draws grows, frequencies tend to move closer to expected proportions. But “closer” does not mean “perfectly equal.”

**Variance**  
Random systems fluctuate. Variation is not automatically signal.

**Randomness illusion**  
Humans expect randomness to look balanced. Real randomness often looks streaky, clustered, and unfair.

**False signal detection**  
A core Data Science skill is not finding patterns. It is testing whether the pattern deserves belief.

In [ ]:
reflection_note = """# Reflection Note — Can You Beat Randomness?

This notebook is not about predicting lottery numbers.

Main lesson:
Randomness produces visible patterns. Hot numbers, cold numbers, and overdue numbers can appear even when the process is fair.

Data Science starts when intuition fails: we count, simulate, compare, and decide whether a pattern is evidence or noise.

Final sentence for students:
Randomness is not intuitive. Data Science starts when intuition fails.
"""
(OUT / "reflection_note.md").write_text(reflection_note)
print(reflection_note)

# Final lesson

Data Science is often not about finding patterns.

It is about proving that some patterns are fake.

Students should leave remembering:

> **Randomness is not intuitive. Data Science starts when intuition fails.**